# Multi-Parameter Regression with ResNet50 (PyTorch)

**Goal**: Predict all 3 parameters (kmin, kmax, sigma) from fractal cube images using regression with separate output heads.

**Memory optimized**: Lazy HDF5 loading, grayscale→RGB conversion on-the-fly.

In [ ]:
from pathlib import Path
import json
import time

import h5py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
candidate_files = [
    Path('../../data/raw/flexible_20260412_120127_128x128_10000.h5'),
]

DATA_FILE = next((p for p in candidate_files if p.exists()), None)
OUTPUT_DIR = Path('../../outputs/multi_parameter/regression')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
LEARNING_RATE = 1e-4
EPOCHS = 100
EARLY_STOP_PATIENCE = 15

KMIN_MIN, KMIN_MAX = 1.0, 32.0
KMAX_MIN, KMAX_MAX = 100.0, 256.0
SIGMA_MIN, SIGMA_MAX = 0.5, 5.0

if DATA_FILE is None:
    raise FileNotFoundError('Dataset not found')
print(f'Dataset: {DATA_FILE}')

## Memory-Efficient Dataset (Lazy Loading)

In [ ]:
class HDF5Dataset(Dataset):
    def __init__(self, h5_path, indices, kmin_values, kmax_values, sigma_values,
                 kmin_min=1.0, kmin_max=32.0, kmax_min=100.0, kmax_max=256.0, sigma_min=0.5, sigma_max=5.0):
        self.h5_path = h5_path
        self.indices = indices
        self.kmin = kmin_values[indices]
        self.kmax = kmax_values[indices]
        self.sigma = sigma_values[indices]
        self.kmin_min, self.kmin_max = kmin_min, kmin_max
        self.kmax_min, self.kmax_max = kmax_min, kmax_max
        self.sigma_min, self.sigma_max = sigma_min, sigma_max
        self._h5_file = None
    
    def _get_h5(self):
        if self._h5_file is None:
            self._h5_file = h5py.File(self.h5_path, 'r')
        return self._h5_file
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        h5 = self._get_h5()
        img_idx = self.indices[idx]
        img = h5['images'][img_idx].astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = img[np.newaxis, :, :]
        y_kmin = (self.kmin[idx] - self.kmin_min) / (self.kmin_max - self.kmin_min)
        y_kmax = (self.kmax[idx] - self.kmax_min) / (self.kmax_max - self.kmax_min)
        y_sigma = (self.sigma[idx] - self.sigma_min) / (self.sigma_max - self.sigma_min)
        y = np.array([y_kmin, y_kmax, y_sigma], dtype=np.float32)
        return torch.from_numpy(img), torch.from_numpy(y)
    
    def __del__(self):
        if self._h5_file is not None:
            self._h5_file.close()

In [ ]:
with h5py.File(DATA_FILE, 'r') as f:
    n_samples = f['images'].shape[0]
    kmin_values = f['parameters/k_min'][:]
    kmax_values = f['parameters/k_max'][:]
    sigma_values = f['parameters/sigma'][:]

print(f'Total samples: {n_samples}')
print(f'kmin range:  [{kmin_values.min():.1f}, {kmin_values.max():.1f}]')
print(f'kmax range:  [{kmax_values.min():.1f}, {kmax_values.max():.1f}]')
print(f'sigma range: [{sigma_values.min():.1f}, {sigma_values.max():.1f}]')

In [ ]:
indices = np.arange(n_samples)
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=SEED)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED)

train_ds = HDF5Dataset(DATA_FILE, train_idx, kmin_values, kmax_values, sigma_values,
                        KMIN_MIN, KMIN_MAX, KMAX_MIN, KMAX_MAX, SIGMA_MIN, SIGMA_MAX)
val_ds = HDF5Dataset(DATA_FILE, val_idx, kmin_values, kmax_values, sigma_values,
                      KMIN_MIN, KMIN_MAX, KMAX_MIN, KMAX_MAX, SIGMA_MIN, SIGMA_MAX)
test_ds = HDF5Dataset(DATA_FILE, test_idx, kmin_values, kmax_values, sigma_values,
                       KMIN_MIN, KMIN_MAX, KMAX_MIN, KMAX_MAX, SIGMA_MIN, SIGMA_MAX)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')

## Model with 3 Output Heads

In [ ]:
class ResNet50MultiRegressor(nn.Module):
    def __init__(self, in_channels=1, pretrained=False):
        super().__init__()
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet50(weights=weights)
        self.backbone.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.shared_head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.kmin_head = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
        self.kmax_head = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
        self.sigma_head = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        shared = self.shared_head(features)
        kmin_pred = self.kmin_head(shared).squeeze(-1)
        kmax_pred = self.kmax_head(shared).squeeze(-1)
        sigma_pred = self.sigma_head(shared).squeeze(-1)
        return torch.stack([kmin_pred, kmax_pred, sigma_pred], dim=1)

model = ResNet50MultiRegressor(in_channels=1, pretrained=False).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
criterion = nn.HuberLoss(delta=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-7)
scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

## Train

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                pred = model(X_batch)
                loss = criterion(pred, y_batch)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    preds, targets = [], []
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        total_loss += loss.item() * len(X_batch)
        preds.append(pred.cpu())
        targets.append(y_batch.cpu())
    preds = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    mae = np.abs(preds - targets).mean()
    return total_loss / len(loader.dataset), mae, preds, targets

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_mae': []}
best_val_loss = float('inf')
patience_counter = 0
start_time = time.time()

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, scaler)
    val_loss, val_mae, _, _ = eval_epoch(model, val_loader, criterion)
    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_mae'].append(val_mae)
    lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch+1:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | MAE: {val_mae:.4f} | LR: {lr:.2e}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), OUTPUT_DIR / 'best_model.pt')
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f'Early stopping at epoch {epoch+1}')
            break

train_time = time.time() - start_time
print(f'\nTraining: {train_time/60:.1f} min')

## Evaluate

In [ ]:
model.load_state_dict(torch.load(OUTPUT_DIR / 'best_model.pt'))
_, _, y_pred_norm, y_true_norm = eval_epoch(model, test_loader, criterion)

y_pred_kmin = y_pred_norm[:, 0] * (KMIN_MAX - KMIN_MIN) + KMIN_MIN
y_true_kmin = y_true_norm[:, 0] * (KMIN_MAX - KMIN_MIN) + KMIN_MIN
y_pred_kmax = y_pred_norm[:, 1] * (KMAX_MAX - KMAX_MIN) + KMAX_MIN
y_true_kmax = y_true_norm[:, 1] * (KMAX_MAX - KMAX_MIN) + KMAX_MIN
y_pred_sigma = y_pred_norm[:, 2] * (SIGMA_MAX - SIGMA_MIN) + SIGMA_MIN
y_true_sigma = y_true_norm[:, 2] * (SIGMA_MAX - SIGMA_MIN) + SIGMA_MIN

mae_kmin = mean_absolute_error(y_true_kmin, y_pred_kmin)
rmse_kmin = np.sqrt(mean_squared_error(y_true_kmin, y_pred_kmin))
r2_kmin = r2_score(y_true_kmin, y_pred_kmin)
mae_kmax = mean_absolute_error(y_true_kmax, y_pred_kmax)
rmse_kmax = np.sqrt(mean_squared_error(y_true_kmax, y_pred_kmax))
r2_kmax = r2_score(y_true_kmax, y_pred_kmax)
mae_sigma = mean_absolute_error(y_true_sigma, y_pred_sigma)
rmse_sigma = np.sqrt(mean_squared_error(y_true_sigma, y_pred_sigma))
r2_sigma = r2_score(y_true_sigma, y_pred_sigma)

print('=== Test Metrics ===')
print(f'\nkmin:')
print(f'  MAE:  {mae_kmin:.3f}')
print(f'  RMSE: {rmse_kmin:.3f}')
print(f'  R2:   {r2_kmin:.4f}')
print(f'\nkmax:')
print(f'  MAE:  {mae_kmax:.3f}')
print(f'  RMSE: {rmse_kmax:.3f}')
print(f'  R2:   {r2_kmax:.4f}')
print(f'\nsigma:')
print(f'  MAE:  {mae_sigma:.3f}')
print(f'  RMSE: {rmse_sigma:.3f}')
print(f'  R2:   {r2_sigma:.4f}')

In [ ]:
results = {
    'kmin': {'mae': float(mae_kmin), 'rmse': float(rmse_kmin), 'r2': float(r2_kmin), 'param_range': [float(KMIN_MIN), float(KMIN_MAX)]},
    'kmax': {'mae': float(mae_kmax), 'rmse': float(rmse_kmax), 'r2': float(r2_kmax), 'param_range': [float(KMAX_MIN), float(KMAX_MAX)]},
    'sigma': {'mae': float(mae_sigma), 'rmse': float(rmse_sigma), 'r2': float(r2_sigma), 'param_range': [float(SIGMA_MIN), float(SIGMA_MAX)]},
    'training_time_min': train_time / 60
}

with open(OUTPUT_DIR / 'test_results.json', 'w') as f:
    json.dump(results, f, indent=2)

with open(OUTPUT_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f)

print('Results saved to', OUTPUT_DIR)